# 60 · Resolution budget: positional noise against replicated continuous information, per coordinate

A coordinate can only add information inside a segment if its positional error inside the segment
is smaller than the spread of positions there. This notebook puts, for every coordinate run so far,
the coordinate's **positional noise** next to the **continuous information that replicated**:

| Side | Measure | Source |
|---|---|---|
| noise | gene-fold noise: Var(z_a − z_b)/2 between coordinates rebuilt from disjoint thirds of the construction genes, per specimen × segment, in segment units and relative to the within-segment spread of positions | notebooks 35 and 36 refits; workstream A's folds or posterior when supplied |
| noise | count-split noise: the same between coordinates built from the two halves of the counts | notebooks 35 and 36 |
| noise | label agreement of the coordinate (share of structures whose reviewed label matches the best cut points) | notebook 57 |
| information | (iv) onset order inside mouse S2 beyond r, between mice and in external mice | notebooks 58 and 59 |
| information | (i) within-segment slope agreement between specimens; (ii) replicated interior extrema; (iii) species boundary shifts | notebook 58 |
| information | beyond-step pathway calls and how many label–coordinate discordance alone produces | notebook 57 |

The relation across coordinates is shown, not tested: there are at most four coordinates.
Cross-coordinate rules from the phase-1 protocol (section 8): an interior extremum counts when it
replicates in both specimens on at least two of SCF13, DPT13 and SCF13-ED, including the equal-depth
refit; a continuous-only statement enters the paper only if it passes on the paper's coordinate and
is not reversed on DPT13.

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--extra-coordinate', action='append', default=[],
                    help='label=coordinate.csv[:column]; fold or posterior noise is read from columns '
                         'named fold_* or *_sd in the same file')
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
extra = list(args.extra_coordinate) + [x for x in os.environ.get('PSEUDOSPACE_EXTRA_COORDINATES', '').split(';') if x]
lcp_default = results_root / 'pt_reconstruction_v3' / 'lcp_coordinate.csv'
if lcp_default.is_file() and not any(x.startswith('lcp=') for x in extra):
    extra.append(f'lcp={lcp_default}')
continuous = results_root / 'pt_pathway_continuous'
nb35 = results_root / 'pt_reconstruction_v2' / 'nb35' / 'stage_cache'
nb36 = results_root / 'pt_reconstruction_v2' / 'nb36' / 'stage_cache'
output = continuous / '60_resolution_budget'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
NOTEBOOK_LOGIC_VERSION = '60.resolution_budget.1'
COLORS = {'scf13': '#2a78d6', 'dpt13': '#eb6834', 'scf13_ed': '#1baf7a', 'lcp': '#eda100'}  # validated categorical order
LABELS = {'scf13': 'SCF13 (principal curve)', 'dpt13': 'DPT13', 'scf13_ed': 'SCF13-ED (equal depth)', 'lcp': 'LCP (workstream A)'}
print('Results folder:', output, '· extra coordinates:', extra or 'none')

## 1 · Positional noise per coordinate, species and segment

In [ ]:
import glob

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display

from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.coordinate_evaluation import fold_noise_variance
from pseudospace.coordinate_inputs import common_support_mask, read_coordinate

saved_pt = ad.read_h5ad(results_root / 'minimal_pt_scfates' / 'cross_species_pt_scfates.h5ad', backed='r').obs.copy()
structure = saved_pt.index.astype(str)
specimen = saved_pt['sample'].astype(str).to_numpy()
segment = saved_pt.segment_class.astype(str).to_numpy()
species = saved_pt.comparison_species.astype(str).to_numpy()


def payload(folder, stage, key):
    files = [f for f in glob.glob(str(folder / f'{stage}__*.npz'))]
    if len(files) != 1:
        return None
    with np.load(files[0], allow_pickle=True) as stored:
        return np.asarray(stored[key], float)


arms = pd.read_csv(results_root / 'pt_reconstruction_v2' / 'nb36' / 'arm_coordinates.csv').set_index('structure_id')
coordinates = {'scf13': arms['SCF13'].reindex(structure).to_numpy(float),
               'dpt13': arms['DPT13'].reindex(structure).to_numpy(float),
               'scf13_ed': arms['SCF13-ED'].reindex(structure).to_numpy(float)}
folds = {'scf13': [payload(nb35, f'refit_fold{i}', 'scfates') for i in range(3)],
         'dpt13': [payload(nb35, f'refit_fold{i}', 'dpt') for i in range(3)],
         'scf13_ed': [payload(nb36, f'arm_scf13ed_fold{i}', 'scfates') for i in range(3)]}
halves = {'scf13': [payload(nb35, f'refit_split_{h}', 'scfates') for h in 'AB'],
          'dpt13': [payload(nb35, f'refit_split_{h}', 'dpt') for h in 'AB'],
          'scf13_ed': [payload(nb36, f'arm_scf13ed_half_{h}', 'scfates') for h in 'AB']}
posterior_sd = {}
for item in extra:
    label, spec = item.split('=', 1)
    path, _, column = spec.partition(':')
    table = pd.read_csv(path)
    id_column = 'structure' if 'structure' in table else 'structure_id'
    table = table.set_index(table[id_column].astype(str)).reindex(structure)
    coordinates[label] = table[column or 'position'].to_numpy(float)
    fold_columns = sorted(c for c in table if c.lower().startswith('fold'))
    folds[label] = [table[c].to_numpy(float) for c in fold_columns] if len(fold_columns) >= 2 else []
    halves[label] = []
    sd_columns = [c for c in table if c.lower().endswith('_sd') or c.lower() in ('sd', 'posterior_sd', 'position_sd')]
    if sd_columns:
        posterior_sd[label] = table[sd_columns[0]].to_numpy(float)
    elif {'posterior_q10', 'posterior_q90'}.issubset(table):  # normal approximation from the 10-90% interval
        posterior_sd[label] = ((table.posterior_q90 - table.posterior_q10) / 2.5631).to_numpy(float)
    # Workstream A's notebook 51 caches per-species refits ('mean' per structure, in the file's species order).
    nb51 = Path(path).parent / 'nb51' / 'stage_cache'
    if label == 'lcp' and nb51.is_dir():
        source = pd.read_csv(path)
        order = {sp: source.index[source.species.eq(sp)] for sp in ('mouse', 'human')}

        def refit(stage_suffix):
            values = pd.Series(np.nan, index=source[id_column].astype(str))
            for sp in ('mouse', 'human'):
                z = payload(nb51, f'lcp_{sp}_{stage_suffix}', 'mean')
                full = payload(nb51, f'lcp_{sp}_full', 'mean')
                if z is None or full is None or len(z) != len(order[sp]):
                    return None
                # Guard: the full refit reproduces the published positions in this order.
                if not np.allclose(full, source.position.to_numpy()[order[sp]]):
                    raise ValueError('Notebook 51 refit order does not match lcp_coordinate.csv.')
                values.iloc[order[sp]] = z
            return values.reindex(structure).to_numpy(float)

        folds[label] = [refit(f'fold{i}') for i in range(3)]
        halves[label] = [refit(f'half_{h}') for h in 'AB']
print('Coordinates:', list(coordinates), '· fold refits:', {k: len([f for f in v if f is not None]) for k, v in folds.items()})


def noise_table(label):
    z = coordinates[label]
    ok = np.isfinite(z)
    keep = ok & common_support_mask(np.where(ok, z, 0), specimen) if ok.all() else ok
    trans = logistic_transitions(z[keep], segment[keep], specimen[keep]).pivot(index='specimen', columns='transition', values='position')
    lo = max(z[keep & (specimen == n)].min() for n in np.unique(specimen))
    hi = min(z[keep & (specimen == n)].max() for n in np.unique(specimen))
    rows = []
    fold_list = [f for f in folds.get(label, []) if f is not None]
    half_list = [h for h in halves.get(label, []) if h is not None]
    fold_var = fold_noise_variance({i: f for i, f in enumerate(fold_list)}, specimen, segment, species)[1].mean(axis=1) \
        if len(fold_list) >= 2 else None
    half_var = fold_noise_variance({i: h for i, h in enumerate(half_list)}, specimen, segment, species)[1].mean(axis=1) \
        if len(half_list) == 2 else None
    for n in np.unique(specimen):
        t12, t23 = trans.loc[n, 'S1→S2'], trans.loc[n, 'S2→S3']
        lengths = {'PT-S1': t12 - lo, 'PT-S2': t23 - t12, 'PT-S3': hi - t23}
        sp = species[specimen == n][0]
        for seg, length in lengths.items():
            m = keep & (specimen == n) & (segment == seg)
            spread = float(np.std(z[m]))
            row = {'coordinate': label, 'species': sp, 'specimen': n, 'segment': seg[-2:], 'segment length': length,
                   'position SD within segment': spread}
            for name, var in (('fold', fold_var), ('count split', half_var)):
                if var is not None and (sp, n, seg) in var.index:
                    sd = float(np.sqrt(var.loc[(sp, n, seg)]))
                    row[f'{name} noise SD (segment units)'] = sd / length if length > 0 else np.nan
                    row[f'{name} noise ÷ spread'] = sd / spread if spread > 0 else np.nan
            if label in posterior_sd:
                sd = float(np.nanmedian(posterior_sd[label][m]))
                row['posterior SD (segment units)'] = sd / length if length > 0 else np.nan
                row['posterior SD ÷ spread'] = sd / spread if spread > 0 else np.nan
            rows.append(row)
    return pd.DataFrame(rows)


NOISE = pd.concat([noise_table(label) for label in coordinates], ignore_index=True)
NOISE.to_csv(tables / 'positional_noise_by_specimen_segment.csv', index=False)
noise_cols = [c for c in NOISE if 'noise' in c or 'posterior' in c]
NOISE_SUMMARY = NOISE.groupby(['coordinate', 'species', 'segment'])[noise_cols].mean()
display(NOISE_SUMMARY.round(2))

## 2 · Replicated continuous information per coordinate

In [ ]:


def read(path, **kwargs):
    return pd.read_csv(path, **kwargs) if Path(path).is_file() else None


info_rows = []
for label in coordinates:
    row = {'coordinate': label}
    d57 = read(continuous / '57_beyond_steps' / label / 'tables' / 'decision_summary.csv', index_col=0)
    if d57 is not None:
        v = d57.value
        row.update({'beyond label steps: calls': float(v['observed test A calls (beyond label steps)']),
                    'beyond label steps: share reproduced by label–coordinate discordance':
                        float(v['primary ratio (arm A test A ÷ observed test A)']),
                    'beyond coordinate steps: calls': float(v['observed test B calls (beyond coordinate steps)']),
                    'label agreement mouse': float(v['label agreement, mouse (mean)']),
                    'label agreement human': float(v['label agreement, human (mean)'])})
    ordering = read(continuous / '58_descriptors' / label / 'tables' / 'iv_onset_ordering.csv')
    if ordering is not None:
        p = ordering[ordering.arm.eq(label) & ordering.pair.eq('mouse') & ordering.segment.eq('S2')
                     & ordering['genes used'].eq('fold-protected')].iloc[0]
        row.update({'(iv) mouse S2 partial ρ': p['partial Spearman (onset | r)'], '(iv) 95% low': p['95% low'],
                    '(iv) 95% high': p['95% high'], '(iv) genes': p['genes']})
    nulls = read(continuous / '58_descriptors' / label / 'tables' / 'observed_against_step_nulls.csv', index_col=0)
    if nulls is not None:
        row['(iv) step-null max'] = nulls.loc['(iv) mouse S2 partial ρ', ['arm A max', 'arm B max']].max()
        for sp in ('mouse', 'human'):
            for seg in ('S1', 'S2', 'S3'):
                key = f'(i) {sp} {seg} slope ρ'
                row[f'(i) {sp} {seg} slope ρ'] = nulls.loc[key, 'observed']
                row[f'(i) {sp} {seg} slope ρ, step-null max'] = nulls.loc[key, ['arm A max', 'arm B max']].max()
    shifts = read(continuous / '58_descriptors' / label / 'tables' / 'iii_species_boundary_shifts.csv', index_col=0)
    if shifts is not None:
        row['(iii) genes shifted beyond registration null'] = int(shifts.passes.sum())
    external = read(continuous / '59_external_onsets' / 'tables' / 'external_onset_replication.csv')
    if external is not None and label in set(external.coordinate):
        e = external.set_index('coordinate').loc[label]
        row.update({'(iv) external partial ρ': e.get('partial Spearman (external | our r)', np.nan),
                    '(iv) external 95% low': e.get('95% low (donor bootstrap)', np.nan),
                    '(iv) external 95% high': e.get('95% high (donor bootstrap)', np.nan)})
    info_rows.append(row)
INFO = pd.DataFrame(info_rows).set_index('coordinate')
display(INFO.T.round(3))

## 3 · Cross-coordinate rules: interior extrema and boundary shifts

In [ ]:
extrema = {label: read(continuous / '58_descriptors' / label / 'tables' / 'ii_replicated_interior_extrema.csv')
           for label in ('scf13', 'dpt13', 'scf13_ed')}
extrema = {k: v for k, v in extrema.items() if v is not None}
if len(extrema) == 3:
    keyed = {label: set(zip(f.gene, f.species, f.segment, f.kind)) for label, f in extrema.items()}
    all_keys = set().union(*keyed.values())
    cross = pd.DataFrame([{'gene': g, 'species': s, 'segment': seg, 'kind': kind,
                           **{label: (g, s, seg, kind) in keyed[label] for label in keyed}} for g, s, seg, kind in all_keys])
    cross['coordinates'] = cross[list(keyed)].sum(axis=1)
    cross['counts (≥ 2 coordinates including SCF13-ED)'] = cross.coordinates.ge(2) & cross.scf13_ed
    cross.sort_values(['species', 'segment', 'gene']).to_csv(tables / 'interior_extrema_across_coordinates.csv', index=False)
    print('Replicated interior extrema meeting the cross-coordinate rule:',
          cross.groupby('species')['counts (≥ 2 coordinates including SCF13-ED)'].sum().to_dict())
    display(cross[cross['counts (≥ 2 coordinates including SCF13-ED)']].sort_values(['species', 'segment']).head(40))
shift_tables = {label: read(continuous / '58_descriptors' / label / 'tables' / 'iii_species_boundary_shifts.csv', index_col=0)
                for label in ('scf13', 'dpt13', 'scf13_ed')}
shift_tables = {k: v for k, v in shift_tables.items() if v is not None}
if 'scf13' in shift_tables and 'dpt13' in shift_tables:
    a, b = shift_tables['scf13'], shift_tables['dpt13']
    passing = a.index[a.passes]
    not_reversed = [g for g in passing if g in b.index and np.sign(b.loc[g, 'shift (segment units)']) == np.sign(a.loc[g, 'shift (segment units)'])]
    print(f'(iii) SCF13 shifted genes {len(passing)}; same sign on DPT13 {len(not_reversed)}')
    pd.DataFrame({'gene': passing, 'same sign on DPT13': [g in not_reversed for g in passing]}).to_csv(
        tables / 'boundary_shifts_scf13_vs_dpt13.csv', index=False)
cand = read(continuous / '57_beyond_steps' / 'within_segment_candidates_across_coordinates.csv', index_col=0)
if cand is not None:
    print('Within-segment pathway candidates (tests A and B, SCF13 and another coordinate):',
          int(cand['SCF13 and at least one other'].sum()))

## 4 · The resolution-budget table and figure

One row per coordinate. Noise columns are means over the two specimens of a species. The figure
has two panels:

- **Left.** For every species × segment cell and coordinate: positional noise relative to the
  within-segment spread of positions (gene folds; x) against the between-specimen agreement of
  within-segment slopes over fold-protected genes (y). Filled markers exceed every step-null
  replicate of notebook 58; open markers do not.
- **Right.** The onset-order test (iv) in mouse S2 against mouse S2 noise, with 95% intervals; grey
  bars show the largest step-null replicate.

In [ ]:
budget = NOISE_SUMMARY.reset_index().pivot_table(index='coordinate', columns=['species', 'segment'],
                                                 values=[c for c in noise_cols if 'segment units' in c])
budget.columns = [f'{a} · {sp} {seg}' for a, sp, seg in budget.columns]
BUDGET = budget.join(INFO, how='outer')
BUDGET.to_csv(tables / 'resolution_budget.csv')
display(BUDGET.T.round(3))

cells = NOISE_SUMMARY.reset_index()
cells['slope ρ'] = [INFO.loc[r.coordinate, f'(i) {r.species} {r.segment} slope ρ'] if r.coordinate in INFO.index else np.nan
                    for r in cells.itertuples()]
cells['step-null max'] = [INFO.loc[r.coordinate, f'(i) {r.species} {r.segment} slope ρ, step-null max']
                          if r.coordinate in INFO.index else np.nan for r in cells.itertuples()]
cells['beyond step nulls'] = cells['slope ρ'] > cells['step-null max']
cells.to_csv(tables / 'noise_against_slope_agreement.csv', index=False)
print('Spearman, noise ÷ spread against slope agreement (all cells):',
      round(cells[['fold noise ÷ spread', 'slope ρ']].dropna().corr(method='spearman').iloc[0, 1], 2),
      '· cells beyond the step nulls by coordinate:', cells.groupby('coordinate')['beyond step nulls'].sum().to_dict())

import matplotlib.pyplot as plt

fig, (ax, bx) = plt.subplots(1, 2, figsize=(10.5, 4), gridspec_kw={'width_ratios': [1.5, 1]})
for label in COLORS:
    sub = cells[cells.coordinate.eq(label)].dropna(subset=['fold noise ÷ spread', 'slope ρ'])
    for r in sub.itertuples():
        ax.scatter(r[cells.columns.get_loc('fold noise ÷ spread') + 1], r[cells.columns.get_loc('slope ρ') + 1],
                   s=70, marker='o' if r.species == 'mouse' else 's', linewidths=2,
                   facecolor=COLORS[label] if r[cells.columns.get_loc('beyond step nulls') + 1] else 'white',
                   edgecolor=COLORS[label], zorder=3)
        ax.annotate(f'{r.species[0]}{r.segment}', (r[cells.columns.get_loc('fold noise ÷ spread') + 1],
                    r[cells.columns.get_loc('slope ρ') + 1]), xytext=(5, 3), textcoords='offset points',
                    fontsize=7, color='#52514e')
    ax.scatter([], [], s=70, color=COLORS[label], label=LABELS[label])
ax.axhline(0, color='#c3c2b7', lw=1, zorder=0)
ax.set_xlabel('Positional noise ÷ within-segment spread (gene folds)', fontsize=8)
ax.set_ylabel('Within-segment slopes: agreement between specimens\n(Spearman, fold-protected genes)', fontsize=8)
ax.set_title('Filled = beyond every step null · m/h = mouse/human', fontsize=9, loc='left')
ax.legend(fontsize=7, frameon=False, loc='lower left')
noise_col = 'fold noise SD (segment units) · mouse S2'
for label, row in BUDGET.iterrows():
    if not np.isfinite(row.get(noise_col, np.nan)) or not np.isfinite(row.get('(iv) mouse S2 partial ρ', np.nan)):
        continue
    x, yv = row[noise_col], row['(iv) mouse S2 partial ρ']
    bx.plot([x, x], [row['(iv) 95% low'], row['(iv) 95% high']], color=COLORS.get(label, '#52514e'), lw=2)
    bx.scatter([x], [yv], s=70, color=COLORS.get(label, '#52514e'), edgecolor='white', linewidth=2, zorder=3)
    bx.scatter([x + .006], [row['(iv) step-null max']], s=40, marker='_', color='#52514e', linewidth=2, zorder=3)
    bx.annotate(f"{LABELS.get(label, label).split(' ')[0]} ({int(row['(iv) genes'])} genes)", (x, yv),
                xytext=(6, 4), textcoords='offset points', fontsize=7, color='#0b0b0b')
bx.axhline(0, color='#c3c2b7', lw=1, zorder=0)
bx.set_xlabel('Mouse S2 positional noise (gene folds, SD in segment units)', fontsize=8)
bx.set_ylabel('Onset order beyond r (partial Spearman)', fontsize=8)
bx.set_title('(iv) onset order; dash = largest step null', fontsize=9, loc='left')
for a in (ax, bx):
    for side in ('top', 'right'):
        a.spines[side].set_visible(False)
fig.tight_layout()
for ext_ in ('pdf', 'png'):
    fig.savefig(figures / f'resolution_budget.{ext_}', dpi=200)
plt.show()
(output / 'run_record.json').write_text(json.dumps({'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
                                                    'coordinates': list(coordinates), 'extra': extra}, indent=2))